# Nornickel — сегментация талька (fast_768)

Обучение **с нуля** → `best_talk.pt`.

Датасет в zip — **2272×1704**; resize **576×768** только при обучении.

1. Kaggle Dataset: `nornickel_talc_segmentation.zip`
2. GPU T4, Run All
3. Скачай `best_talk.pt` → `models/weights/`

In [ ]:
!pip -q install segmentation-models-pytorch albumentations

In [ ]:
# --- fast_768 (фиксированный пресет) ---
EXPERIMENT = "fast_768"
IMG_H, IMG_W = 576, 768
BATCH_SIZE = 4
MAX_EPOCHS = 80
LR = 3e-4
ENCODER = "efficientnet-b4"
ENCODER_WEIGHTS = "imagenet"
EARLY_STOP_PATIENCE = 12
PANO_SAMPLE_WEIGHT = 3.0
SEED = 42

print(f"Train {IMG_H}x{IMG_W}, batch={BATCH_SIZE}, early_stop={EARLY_STOP_PATIENCE}")

In [ ]:
import os
import random
import shutil
import sys
from pathlib import Path

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import segmentation_models_pytorch as smp
from tqdm import tqdm


def find_data_root() -> Path:
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for meta_path in kaggle_input.rglob("metadata.csv"):
            root = meta_path.parent
            if (root / "images").is_dir() and (root / "masks").is_dir():
                return root
        raise FileNotFoundError("talc_segmentation not found under /kaggle/input")
    return Path("../dataset/kaggle/talc_segmentation")


def prepare_data_root(src: Path, work_dir: Path) -> Path:
    if not Path("/kaggle/input").is_dir():
        return src
    cache = work_dir / "dataset_cache"
    marker = cache / ".ready"
    if not marker.is_file():
        if cache.exists():
            shutil.rmtree(cache)
        print(f"Copying dataset -> {cache}")
        shutil.copytree(src, cache)
        marker.touch()
    return cache


WORK_DIR = Path("/kaggle/working") if Path("/kaggle/input").exists() else Path("../dataset/kaggle/runs")
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT = prepare_data_root(find_data_root(), WORK_DIR)
CKPT_PATH = WORK_DIR / "best_talk.pt"
HISTORY_PATH = WORK_DIR / "history_fast_768.csv"
PREVIEW_PATH = WORK_DIR / "val_preview_fast_768.png"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

NUM_WORKERS = min(4, os.cpu_count() or 1) if sys.platform != "win32" else 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("Data:", DATA_ROOT)
print("Output:", WORK_DIR)

In [ ]:
meta = pd.read_csv(
    DATA_ROOT / "metadata.csv",
    dtype={"sample_id": str, "cvat_filename": str, "kind": str, "split": str},
)
print(meta["split"].value_counts())
print(meta["kind"].value_counts())
meta.head()

In [ ]:
def load_rgb(path: Path) -> np.ndarray:
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def load_mask(path: Path) -> np.ndarray:
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(path)
    return (mask > 127).astype(np.float32)


def build_train_transform():
    """Аугментации: геометрия + тёмные/контрастные варианты (CLAHE, gamma)."""
    return A.Compose([
        A.Resize(IMG_H, IMG_W),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
            scale=(0.85, 1.15),
            rotate=(-20, 20),
            p=0.5,
        ),
        A.Resize(IMG_H, IMG_W),
        A.RandomBrightnessContrast(brightness_limit=0.35, contrast_limit=0.35, p=0.6),
        A.RandomGamma(gamma_limit=(60, 160), p=0.5),
        A.CLAHE(clip_limit=(2.0, 6.0), tile_grid_size=(8, 8), p=0.35),
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=20, val_shift_limit=20, p=0.3),
        A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
        A.Normalize(),
        ToTensorV2(),
    ])


val_tf = A.Compose([
    A.Resize(IMG_H, IMG_W),
    A.Normalize(),
    ToTensorV2(),
])


def preload_samples(df: pd.DataFrame) -> list[tuple[np.ndarray, np.ndarray]]:
    images_dir = DATA_ROOT / "images"
    masks_dir = DATA_ROOT / "masks"
    out = []
    for _, row in df.iterrows():
        cvat_name = str(row["cvat_filename"])
        stem = Path(cvat_name).stem
        out.append((load_rgb(images_dir / cvat_name), load_mask(masks_dir / f"{stem}.png")))
    return out


class TalcDataset(Dataset):
    def __init__(self, df, transform, samples):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image, mask = self.samples[idx]
        out = self.transform(image=image.copy(), mask=mask.copy())
        return out["image"], out["mask"].unsqueeze(0)


train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "val"].reset_index(drop=True)

print("Preloading train/val into RAM ...")
train_samples = preload_samples(train_df)
val_samples = preload_samples(val_df)

train_ds = TalcDataset(train_df, build_train_transform(), train_samples)
val_ds = TalcDataset(val_df, val_tf, val_samples)

sample_weights = [
    PANO_SAMPLE_WEIGHT if row["kind"] == "panorama_tile" else 1.0
    for _, row in train_df.iterrows()
]
train_sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

loader_kw = {"batch_size": BATCH_SIZE, "pin_memory": torch.cuda.is_available(), "num_workers": NUM_WORKERS}
if NUM_WORKERS > 0:
    loader_kw["persistent_workers"] = True
    loader_kw["prefetch_factor"] = 2

train_loader = DataLoader(train_ds, sampler=train_sampler, shuffle=False, **loader_kw)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kw)

xb, yb = next(iter(train_loader))
assert xb.shape[2:] == (IMG_H, IMG_W), xb.shape
len(train_ds), len(val_ds)

In [ ]:
model = smp.UnetPlusPlus(
    encoder_name=ENCODER,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=3,
    classes=1,
    activation=None,
).to(DEVICE)

class FocalTverskyLoss(nn.Module):
    """Focal Tversky для бинарной маски; не зависит от версии smp."""

    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        targets = targets.float()
        dims = (2, 3)
        tp = (probs * targets).sum(dim=dims)
        fp = (probs * (1 - targets)).sum(dim=dims)
        fn = ((1 - probs) * targets).sum(dim=dims)
        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        return ((1 - tversky) ** self.gamma).mean()


tversky = FocalTverskyLoss(alpha=0.7, beta=0.3, gamma=0.75)
bce = nn.BCEWithLogitsLoss()


def combined_loss(logits, targets):
    return 0.6 * tversky(logits, targets) + 0.4 * bce(logits, targets)


optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

f"params_M={sum(p.numel() for p in model.parameters()) / 1e6:.1f}"

In [ ]:
@torch.no_grad()
def batch_iou_dice(logits, targets, thresh=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > thresh).float()
    inter = (preds * targets).sum(dim=(1, 2, 3))
    union = ((preds + targets) > 0).float().sum(dim=(1, 2, 3))
    iou = ((inter + 1e-6) / (union + 1e-6)).mean().item()
    dice = ((2 * inter + 1e-6) / (preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + 1e-6)).mean().item()
    return iou, dice


@torch.no_grad()
def eval_by_kind(loader, df_val):
    """Dice отдельно для ch1 и pano на val."""
    model.eval()
    per_kind = {}
    for start in range(0, len(val_ds), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(val_ds))
        xs, ys = [], []
        for i in range(start, end):
            x, y = val_ds[i]
            xs.append(x)
            ys.append(y)
        images = torch.stack(xs).to(DEVICE)
        masks = torch.stack(ys).to(DEVICE)
        logits = model(images)
        _, dice = batch_iou_dice(logits, masks)
        for j, idx in enumerate(range(start, end)):
            kind = df_val.iloc[idx]["kind"]
            logit_j = logits[j : j + 1]
            mask_j = masks[j : j + 1]
            _, d = batch_iou_dice(logit_j, mask_j)
            per_kind.setdefault(kind, []).append(d)
    return {k: float(np.mean(v)) for k, v in per_kind.items()}


def run_epoch(loader, train: bool):
    model.train(train)
    total_loss = total_iou = total_dice = n = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, masks in loader:
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            if train:
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    logits = model(images)
                    loss = combined_loss(logits, masks)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(images)
                loss = combined_loss(logits, masks)
            iou, dice = batch_iou_dice(logits, masks)
            bs = images.size(0)
            total_loss += loss.item() * bs
            total_iou += iou * bs
            total_dice += dice * bs
            n += bs
    return total_loss / n, total_iou / n, total_dice / n


best_dice = 0.0
best_epoch = 0
no_improve = 0
history = []

for epoch in tqdm(range(1, MAX_EPOCHS + 1), desc=EXPERIMENT):
    tr_loss, tr_iou, tr_dice = run_epoch(train_loader, train=True)
    va_loss, va_iou, va_dice = run_epoch(val_loader, train=False)
    scheduler.step()
    kind_dice = eval_by_kind(val_loader, val_df)

    row = {
        "epoch": epoch,
        "train_loss": tr_loss,
        "val_loss": va_loss,
        "val_iou": va_iou,
        "val_dice": va_dice,
        **{f"val_dice_{k}": v for k, v in kind_dice.items()},
    }
    history.append(row)

    kind_str = ", ".join(f"{k}={v:.3f}" for k, v in kind_dice.items())
    tqdm.write(
        f"Ep {epoch:02d} | loss {tr_loss:.3f}/{va_loss:.3f} | "
        f"val Dice {va_dice:.3f} IoU {va_iou:.3f} | {kind_str}"
    )

    if va_dice > best_dice:
        best_dice = va_dice
        best_epoch = epoch
        no_improve = 0
        torch.save(
            {
                "experiment": EXPERIMENT,
                "model_state": model.state_dict(),
                "encoder": ENCODER,
                "img_size": (IMG_H, IMG_W),
                "val_dice": va_dice,
                "val_iou": va_iou,
                "kind_dice": kind_dice,
                "epoch": epoch,
            },
            CKPT_PATH,
        )
    else:
        no_improve += 1
        if no_improve >= EARLY_STOP_PATIENCE:
            tqdm.write(f"Early stop at epoch {epoch} (best={best_dice:.3f} @ ep {best_epoch})")
            break

pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
print(f"Best val Dice: {best_dice:.4f} (epoch {best_epoch})")
print(f"Checkpoint: {CKPT_PATH}")

In [ ]:
hist = pd.read_csv(HISTORY_PATH)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
hist.plot(x="epoch", y=["train_loss", "val_loss"], ax=ax[0], title=f"Loss ({EXPERIMENT})")
cols = [c for c in hist.columns if c.startswith("val_dice")]
hist.plot(x="epoch", y=cols, ax=ax[1], title="Val Dice")
plt.tight_layout()
plt.savefig(WORK_DIR / "curves_fast_768.png", dpi=120)
plt.show()

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()


def denorm(img_t):
    x = img_t.cpu().numpy().transpose(1, 2, 0)
    x = x * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    return np.clip(x, 0, 1)


n_show = min(4, len(val_ds))
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
if n_show == 1:
    axes = np.expand_dims(axes, 0)

for i in range(n_show):
    x, y = val_ds[i]
    kind = val_df.iloc[i]["kind"]
    with torch.no_grad():
        pred = torch.sigmoid(model(x.unsqueeze(0).to(DEVICE)))[0, 0].cpu().numpy()
    gt = y[0].numpy()
    img = denorm(x)
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"{kind}")
    axes[i, 1].imshow(gt, cmap="magma")
    axes[i, 1].set_title("GT")
    axes[i, 2].imshow(pred, cmap="magma", vmin=0, vmax=1)
    axes[i, 2].set_title("Pred")
    overlay = img.copy()
    overlay[pred > 0.5] = overlay[pred > 0.5] * 0.5 + np.array([1.0, 0.3, 0.1]) * 0.5
    axes[i, 3].imshow(overlay)
    axes[i, 3].set_title("Overlay")
    for ax in axes[i]:
        ax.axis("off")

plt.suptitle(f"{EXPERIMENT} | best Dice={ckpt['val_dice']:.3f} ep={ckpt['epoch']}")
plt.tight_layout()
plt.savefig(PREVIEW_PATH, dpi=120)
plt.show()
print("Saved:", PREVIEW_PATH)

## Output

- `best_talk.pt` → положи в `models/weights/`
- `history_fast_768.csv`, `val_preview_fast_768.png`